In [256]:
import pandas as pd

In [257]:
from pprint import pprint

In [258]:
pd.set_option('display.float_format', '{:.4f}'.format)

In [259]:
const = { 'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
pprint(const)

{'Количество_человек': 2000000,
 'НДФЛ': 0.13,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374}


In [260]:
dynamic = pd.read_excel(
    "Модель по ОФЗ ИН (в) переделыш.xlsx",
    sheet_name="Входные данные",    # имя листа
    usecols="A:C",                 #  параметр, который говорит: «бери только эти столбцы»
    skiprows=0                     # сколько строк пропустить в начале файла (при чтении)
)
dynamic.columns = ['Год', 'Ставка инфляции', 'Ставка депозита']
dynamic = dynamic.dropna(subset=['Год'])
dynamic['Год'] =  dynamic['Год'].astype(int)
dynamic


,Год,Ставка инфляции,Ставка депозита
0,2026,0.0560,0.1306
1,2027,0.0400,0.0800
2,2028,0.0400,0.0690


In [261]:
ofz = dynamic.copy()
ofz

,Год,Ставка инфляции,Ставка депозита
0,2026,0.0560,0.1306
1,2027,0.0400,0.0800
2,2028,0.0400,0.0690


In [262]:
ofz['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz['Количество человек'] = const['Количество_человек']
ofz['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0560,0.1306,380000000000,2000000,0.0250
1,2027,0.0400,0.0800,380000000000,2000000,0.0250
2,2028,0.0400,0.0690,380000000000,2000000,0.0250


In [263]:
ofz['На руках у человека, руб'] = ofz['Привлекаемые средства'] / ofz['Количество человек']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000


In [264]:
ofz['Облигаций штук'] = ofz['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000


In [265]:
ofz['Инфляционный множитель']= (1 + dynamic['Ставка инфляции']).cumprod()
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422


In [266]:
ofz['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz['Инфляционный множитель']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960


In [267]:
ofz['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz.loc[ofz.index > 0, 'Номинал на начало'] = ofz['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000


In [268]:
ofz['Индексация номинала'] = ofz['Номинал на начало'] * ofz['Ставка инфляции']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960


In [269]:
ofz['Купон, руб'] = ofz['Номинал после индексации'] * ofz['Ставка купона']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424


In [270]:
ofz['Доход без вычета'] = ofz['Купон, руб'] * ofz['Облигаций штук']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056


In [271]:
ofz ['Налоговый вычет, руб']= ofz['На руках у человека, руб'] * const['НДФЛ']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000,24700.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400,24700.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056,24700.0000


In [272]:
ofz['Доход с вычетом'] = ofz['Налоговый вычет, руб'] + ofz['Доход без вычета']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000,24700.0000,29716.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400,24700.0000,29916.6400
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056,24700.0000,30125.3056


In [273]:
cols = ofz.columns.tolist()
cols

['Год',
 'Ставка инфляции',
 'Ставка депозита',
 'Привлекаемые средства',
 'Количество человек',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал после индексации',
 'Номинал на начало',
 'Индексация номинала',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']

In [274]:
df = ofz[['Год',
 'Привлекаемые средства',
 'Количество человек',
 'Ставка инфляции',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]
df

,Год,Привлекаемые средства,Количество человек,Ставка инфляции,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал на начало,Индексация номинала,Номинал после индексации,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,380000000000,2000000,0.0560,0.0250,190000.0000,19.0000,1.0560,10000.0000,560.0000,10560.0000,264.0000,5016.0000,24700.0000,29716.0000
1,2027,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.0982,10560.0000,422.4000,10982.4000,274.5600,5216.6400,24700.0000,29916.6400
2,2028,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1422,10982.4000,439.2960,11421.6960,285.5424,5425.3056,24700.0000,30125.3056
